# Chapter 13 — Loading and Preprocessing Data with TensorFlow
Companion notebook to `part2_deep_learning/13_loading_preprocessing_data.md`.

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras

## tf.data.Dataset basics: chaining transformations

In [ ]:
dataset = tf.data.Dataset.from_tensor_slices(tf.range(10))
dataset = dataset.repeat(3).batch(7)
for item in dataset:
    print(item)

In [ ]:
dataset2 = dataset.map(lambda x: x * 2)
dataset2 = dataset2.filter(lambda x: tf.reduce_sum(x) < 50)
for item in dataset2.take(2):
    print(item)

## Shuffling + a full pipeline pattern

In [ ]:
dataset3 = tf.data.Dataset.range(10).repeat(3)
dataset3 = dataset3.shuffle(buffer_size=5, seed=42).batch(7)
for item in dataset3:
    print(item)

In [ ]:
# A typical end-to-end pipeline over an in-memory numpy array (stand-in for reading from files)
X = np.random.rand(1000, 8).astype(np.float32)
y = np.random.randint(0, 2, size=(1000,)).astype(np.int32)

full_dataset = tf.data.Dataset.from_tensor_slices((X, y))
full_dataset = (full_dataset
                .cache()
                .shuffle(buffer_size=1000)
                .batch(32)
                .prefetch(tf.data.experimental.AUTOTUNE))

model = keras.models.Sequential([
    keras.layers.Dense(16, activation="relu", input_shape=[8]),
    keras.layers.Dense(1, activation="sigmoid"),
])
model.compile(loss="binary_crossentropy", optimizer="adam", metrics=["accuracy"])
model.fit(full_dataset, epochs=3)   # Keras accepts a Dataset directly!

## Embeddings for high-cardinality categorical features

In [ ]:
vocab_size = 500       # e.g. 500 possible zip codes
embedding_dim = 8

embedding_layer = keras.layers.Embedding(input_dim=vocab_size, output_dim=embedding_dim)

sample_ids = tf.constant([3, 47, 210, 499])
embedded = embedding_layer(sample_ids)
print("Embedded shape:", embedded.shape)   # (4, 8) -- one 8-dim trainable vector per category

## Keras preprocessing layer: Normalization

In [ ]:
normalization_layer = keras.layers.Normalization()
normalization_layer.adapt(X)   # learn mean/std from data, like a Scaler's .fit()

norm_model = keras.models.Sequential([
    normalization_layer,
    keras.layers.Dense(16, activation="relu"),
    keras.layers.Dense(1, activation="sigmoid"),
])
norm_model.compile(loss="binary_crossentropy", optimizer="adam", metrics=["accuracy"])
norm_model.fit(X, y, epochs=3)

## ✍️ Practice
1. Build a `tf.data` pipeline over several CSV-like numpy chunks (interleave + shuffle + map + batch + prefetch).
2. Why should `.cache()` generally go before `.shuffle()`/`.repeat()`, not after?
3. Compare model size/training speed: one-hot encoding a 500-category feature vs an Embedding(output_dim=16).
4. Debug: `fit()` throughput looks fine but GPU utilization is only 40% — what's the likely bottleneck, first fix to try?

In [ ]:
# Your code / notes here